In [ ]:
#Step 1: Unzip the images
import shutil
shutil.unpack_archive("images.zip")

In [ ]:
#Importing all the libraries
import pandas as pd
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential, load_model
from keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout
import matplotlib.pyplot as plt
import tensorflow.keras as keras
from tensorflow.keras import layers

In [ ]:
## Step 2: Load and preprocess the data.
# Load the labels from labels.csv
labels_df = pd.read_csv("labels.csv", sep=',', header=None)
labels_df.columns = ['image_id', 'class', 'x_min', 'y_min', 'x_max', 'y_max']
labels_df.head()

In [ ]:
# Adjust the image IDs in the dataframe
labels_df['image_id'] = labels_df['image_id'].apply(lambda x: f"{x:08d}") # 1 : 00000001
labels_df.head()

In [ ]:
labels_df.info()

In [ ]:
#Use iloc to pick the first
labels_df = labels_df.iloc[:3000]

In [ ]:
labels_df.info()

In [ ]:
#Load the correnponding images from Images folder
images_folder = "Images/"
images=[]

for index, row in labels_df.iterrows():
    img_path = os.path.join(images_folder, f"{row['image_id']}.jpg")
    img = cv2.imread(img_path)
    if img is not None:
        images.append(img)
    else:
        print(f"Failed to load image: {img_path}")



#Check if the images are loaded
if(len(images) == 0):
    print("No images loaded. Please check the image paths")
else:
    print(f"{len(images)} images loaded successfully")

In [ ]:
#Analyze the distribution of vehicle types in the limited dataset
vehicle_types = labels_df['class'].value_counts()
vehicle_types

In [ ]:
# Adddress data quality issues arising from the discripency between labels and actual file names.
labels_df = labels_df.sort_values('image_id')

In [ ]:
labels_df.head()

In [ ]:
# Resize the images to a common pixel size.
if (len(images))> 0:
  processed_images = [ cv2.resize(img, (224,224)) for img in images]
  processed_images = np.array(processed_images)
  print('Images resized successfully')
else:
  print('No images to resize')


In [ ]:
#Convert the labels class to numerical values
labels = labels_df['class'].to_numpy()
bounding_boxes = labels_df[['x_min','y_min','x_max','y_max']].to_numpy()


In [ ]:
bounding_boxes[:10]

In [ ]:
#Performing Label Encoding
unique_labels = np.unique(labels)
label_to_index = {label : index for index, label in enumerate(unique_labels)}
index_to_label = {index : label for index, label in enumerate(unique_labels)}
labels= np.array([label_to_index[label] for label in labels])

In [ ]:
labels[:10]

In [ ]:
index_to_label

In [ ]:
label_to_index

In [ ]:
# Prepare the data by performing train test split.
X_train, X_test, y_train, y_test, bbox_train, bbox_test =  train_test_split(processed_images, labels, bounding_boxes, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
#Create the CUstom Model
def create_model(input_shape, num_classes):
  inputs = keras.Input(shape=input_shape)
  x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
  x = layers.MaxPooling2D(pool_size=(2,2))(x)
  x = layers.Conv2D(64, (3,3), activation='relu')(x)
  x = layers.MaxPooling2D(pool_size=(2,2))(x)
  x = layers.Conv2D(64, (3,3), activation='relu')(x)
  x=  layers.Flatten()(x)
  x=  layers.Dense(64, activation='relu')(x)
  layers.Dense(num_classes, activation='softmax')(x)
  vehicle_class = layers.Dense(num_classes, activation='softmax', name='vehicle_class')(x)
  bounding_box = layers.Dense(4,  name='bounding_box')(x)

  model = keras.Model(inputs=inputs, outputs=[vehicle_class, bounding_box])
  return model


In [ ]:
#Build Pre Trained Model
def build_pretrained_model(based_model_class, input_shape, num_class): # 224,224,3
  base_model = based_model_class(weights='imagenet', include_top=False, input_shape=input_shape)

  x= GlobalAveragePooling2D()(base_model.output) # Converts the 3dFeature Map --> 1D feature vector.
  vehicle_class =layers.Dense(num_classes, activation='softmax', name='vehicle_class')(x)
  bounding_box = layers.Dense(4, name='bounding_box')(x)

  model = keras.Model(inputs=base_model.input, outputs=[vehicle_class, bounding_box])
  return model


In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, EfficientNetB0
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

input_shape = processed_images[0].shape
print(input_shape)

In [ ]:
num_classes=len(unique_labels)
num_classes

In [ ]:
#Define out Pretrained Models
models = {
    'Custom CNN' : create_model(input_shape, num_classes),
    'VGG16' : build_pretrained_model(VGG16, input_shape, num_classes),
    'ResNet50' : build_pretrained_model(ResNet50, input_shape, num_classes),
    'MobileNet' : build_pretrained_model(MobileNet, input_shape, num_classes),
    'EfficientNetB0' : build_pretrained_model(EfficientNetB0, input_shape, num_classes)
}


In [ ]:
results = {}
for model_name, model in models.items():
  print(f'Training {model_name}...')
  model.compile(optimizer='adam', loss= {'vehicle_class' : 'sparse_categorical_crossentropy', 'bounding_box' : 'mse'},
                metrics={'vehicle_class' : 'accuracy' , 'bounding_box' : 'mse'})

  model.fit(X_train, {'vehicle_class' : y_train, 'bounding_box' : bbox_train}, epochs=20, validation_data= (X_test, {'vehicle_class': y_test, 'bounding_box': bbox_test}) )

   #Evaluate the model and check the test Results
  test_results = model.evaluate(X_test, {'vehicle_class': y_test, 'bounding_box': bbox_test}, verbose=2)
  results[model_name] = {'test_metrics' : test_results, }

  print('\nTest Results:', test_results)
  print(f"Finished Training {model_name}. ")

In [ ]:
#Compare Models
import matplotlib.pyplot as plt
comparision_table  = []
for model_name, result in results.items():
  test_metrics = result['test_metrics']
  comparision_table.append({
      'Model' : model_name,
      'Classification Accuracy' : test_metrics[4],
      'Bounding Box MAE' : test_metrics[3]
  })

comparision_df = pd.DataFrame(comparision_table)
print("Model Comparision: ")
print(comparision_df)

#Visualize the results
comparision_df.plot(x= 'Model', kind='bar', figsize=(10,6))
plt.xlabel('Model')
plt.ylabel('Metric Value')
plt.title('Model Performance Comparision')
plt.show()

In [ ]:
#Run inference on the Image
import matplotlib.pyplot as plt

#Choose a few sample images for inference
sample_images = X_test[:5] # Adjust the number of sample images as needed

#Perform inference on the sample images

for model_name, model in models.items():
  print('Predicting Mdoel: ',model_name)
  print('--------------------------------')
  predictions = model.predict(sample_images)

  #Extract the predicted bounding box coordinates
  predicted_bounding_boxes = predictions[1]

  #Visualize the sample images with predicted bounding boxes
  for i in range(len(sample_images)):
    plt.figure()
    plt.imshow(sample_images[i])
    plt.gca().add_patch(plt.Rectangle((predicted_bounding_boxes[i][0], predicted_bounding_boxes[i][1]),
                                      predicted_bounding_boxes[i][2] - predicted_bounding_boxes[i][0],
                                      predicted_bounding_boxes[i][3] - predicted_bounding_boxes[i][1],
                                      edgecolor='r', linewidth=2,
                                      fill=False))
    plt.show()

